# Documentation du Dashboard — Robot Quadrupède

**Projet :** QuadBot — Robot quadrupède autonome  
**Composants :** ESP32 · PCA9685 · 8 servo-moteurs · Capteur MQ-9  
**Interface :** Dashboard HTML/JS communiquant via WebSocket

---

## Architecture générale

```
┌─────────────┐        WebSocket :81        ┌──────────────┐
│  Navigateur │ ◄─────────────────────────► │    ESP32     │
│  dashboard  │   JSON {type, cmd, speed}   │  WiFi + WS   │
│   HTML/JS   │ ◄── JSON {type, sensor...}  │  + MQ-9 ADC  │
└─────────────┘                             └──────┬───────┘
                                                   │ I2C
                                            ┌──────▼───────┐
                                            │   PCA9685    │
                                            │  ch0 → ch7   │
                                            └──────┬───────┘
                                                   │ PWM
                                            ┌──────▼───────┐
                                            │  8 servos    │
                                            │  (2/membre)  │
                                            └──────────────┘
```

### Légende des tags utilisés dans ce document

| Tag | Signification |
|-----|---------------|
| `[WS]` | Échange WebSocket avec l'ESP32 |
| `[ESP]` | Donnée reçue de l'ESP32 |
| `[UI]` | Mise à jour purement visuelle |

---
## 1. `connect()` — Connexion à l'ESP32 `[WS]` `[UI]`

### Rôle

C'est la **fonction centrale** du dashboard. Elle lit l'adresse IP saisie par l'utilisateur, ouvre une connexion WebSocket persistante vers l'ESP32, et attache tous les handlers d'événements réseau.

Un WebSocket est différent d'une requête HTTP classique : la connexion reste **ouverte en permanence** dans les deux sens. L'ESP peut envoyer des données à tout moment sans que le navigateur les ait demandées, et inversement.

### Déclenchement

Appelée manuellement par l'utilisateur via le bouton **CONN.** après avoir saisi l'IP de l'ESP32.

### Code annoté

```javascript
function connect() {
  // 1. Lit l'IP saisie dans le champ texte (ex: "192.168.1.100")
  const ip = document.getElementById('ip-input').value.trim();

  // 2. Si une connexion existe déjà, on la ferme proprement
  //    avant d'en ouvrir une nouvelle
  if (ws) ws.close();

  // 3. Ouverture du WebSocket
  //    ws://  = WebSocket non-chiffré (équivalent http://)
  //    :81    = port d'écoute WebSocket sur l'ESP32
  //    /ws    = endpoint WebSocket déclaré dans le code C++
  ws = new WebSocket(`ws://${ip}:81/ws`);

  // 4. Connexion établie avec succès
  ws.onopen = () => {
    setStatus(true);
    log('WebSocket connecté', 'entry');
    // Lance un ping toutes les 2 secondes pour mesurer la latence
    pingInterval = setInterval(sendPing, 2000);
  };

  // 5. Connexion perdue (WiFi coupé, ESP redémarré...)
  ws.onclose = () => {
    setStatus(false);
    clearInterval(pingInterval);  // Arrête les pings
    clearInterval(sendInterval);  // Arrête l'envoi des commandes moteur
    log('WebSocket déconnecté', 'err');
  };

  ws.onerror = () => log('Erreur WebSocket', 'err');

  // 6. Message reçu de l'ESP32 — routage selon le champ "type"
  ws.onmessage = (e) => {
    try {
      const d = JSON.parse(e.data);       // Décode le JSON reçu

      if (d.type === 'pong') {
        // Réponse à notre ping → calcul de la latence en ms
        document.getElementById('latency').textContent =
          (Date.now() - pingTime) + ' ms';
        return;
      }

      if (d.type === 'sensor') updateSensors(d); // Données MQ-9

    } catch(_) {}  // Ignore les messages non-JSON
  };
}
```

### États du WebSocket (`ws.readyState`)

| Valeur | Constante | Signification |
|--------|-----------|---------------|
| `0` | `CONNECTING` | Connexion en cours |
| `1` | `OPEN` | Connexion active |
| `2` | `CLOSING` | Fermeture en cours |
| `3` | `CLOSED` | Connexion fermée |

> **Pourquoi le port 81 ?**  
> L'ESP32 fait tourner deux serveurs en parallèle : HTTP sur le port **80** et WebSocket sur le port **81**. Séparer les ports évite les conflits et simplifie le code C++.

> **Causes d'échec fréquentes :**  
> IP incorrecte · ESP non connecté au même réseau WiFi · Pare-feu bloquant le port 81

---
## 2. `sendCmd(cmd)` — Envoi d'une commande moteur `[WS]`

### Rôle

Traduit un appui sur une flèche en **mouvement réel des servos**. Elle construit un message JSON et l'envoie via WebSocket à l'ESP32, qui l'interprète pour piloter le PCA9685 via I2C.

### Code annoté

```javascript
function sendCmd(cmd) {
  // Vérifie que le WebSocket est ouvert (état 1 = OPEN)
  if (ws?.readyState !== 1) {
    log('Non connecté', 'warn');
    return;
  }

  // Lit la valeur du slider de vitesse (0 à 100)
  const speed = parseInt(document.getElementById('speed').value);

  // Construit et envoie le message JSON
  //   type  → identifie le type de message (l'ESP filtre sur ce champ)
  //   cmd   → la direction : forward / backward / left / right / stop
  //   speed → entier 0-100, l'ESP l'utilise pour le délai entre frames de marche
  ws.send(JSON.stringify({ type: 'cmd', cmd, speed }));

  log(`> CMD ${cmd.toUpperCase()} speed=${speed}`, 'cmd');
}
```

### Structure du JSON envoyé à l'ESP

```json
{
  "type":  "cmd",
  "cmd":   "forward",
  "speed": 50
}
```

| Champ | Type | Valeurs possibles |
|-------|------|-------------------|
| `type` | `string` | Toujours `"cmd"` |
| `cmd` | `string` | `forward` · `backward` · `left` · `right` · `stop` |
| `speed` | `int` | `0` à `100` — mappé côté ESP sur le délai entre frames |

> **Pourquoi envoyer en boucle ?**  
> `sendCmd` est appelée une première fois dès l'appui, puis répétée toutes les **150 ms** par `startCmd()` tant que le bouton est maintenu. Sans cette répétition, un seul paquet perdu sur le WiFi suffirait à bloquer le robot en pleine marche.

---
## 3. `startCmd(cmd, el)` et `stopCmd()` — Gestion de l'appui `[UI]` `[WS]`

### Rôle

Ces deux fonctions forment une paire. `startCmd` démarre le mouvement quand on *appuie*, `stopCmd` l'arrête quand on *relâche*. Ce comportement **"maintenu = en mouvement"** imite une télécommande physique.

### `startCmd` — Code annoté

```javascript
function startCmd(cmd, el) {
  // Évite de relancer si la même commande est déjà active
  if (currentCmd === cmd) return;

  stopCmd();       // Nettoie toute commande précédente

  currentCmd = cmd;
  el.classList.add('pressed');  // Effet visuel sur le bouton (fond bleu)
  activateLegs(cmd);            // Anime les membres sur la silhouette SVG

  sendCmd(cmd);  // 1er envoi immédiat

  // Puis répétition toutes les 150 ms tant que le bouton est maintenu
  // L'ESP reçoit un flux continu et peut continuer la séquence de marche
  if (cmd !== 'stop') {
    sendInterval = setInterval(() => sendCmd(cmd), 150);
  }
}
```

### `stopCmd` — Code annoté

```javascript
function stopCmd() {
  if (!currentCmd || currentCmd === 'stop') return;

  clearInterval(sendInterval);  // Arrête la répétition des commandes

  // Retire l'effet visuel de tous les boutons
  document.querySelectorAll('.btn').forEach(b => b.classList.remove('pressed'));

  // Remet tous les membres en position passive sur la silhouette
  document.querySelectorAll('.leg').forEach(l => {
    l.classList.remove('active');
    l.classList.add('leg-idle');
  });

  document.getElementById('current-cmd').textContent = 'EN ATTENTE';
  currentCmd = null;

  sendCmd('stop'); // Envoie "stop" à l'ESP pour immobiliser les servos
}
```

### Branchement des événements sur les boutons

```javascript
document.querySelectorAll('.btn[data-cmd]').forEach(btn => {
  const cmd = btn.dataset.cmd;

  // mousedown = appui souris | touchstart = appui tactile (mobile)
  btn.addEventListener('mousedown',  e => { e.preventDefault(); startCmd(cmd, btn); });
  btn.addEventListener('touchstart', e => { e.preventDefault(); startCmd(cmd, btn); },
                       { passive: false });

  // Sauf STOP : il reste actif même après le relâchement
  if (cmd !== 'stop') {
    btn.addEventListener('mouseup',    () => stopCmd());
    btn.addEventListener('mouseleave', () => stopCmd()); // Si la souris glisse hors du bouton
    btn.addEventListener('touchend',   () => stopCmd());
  }
});
```

> **Pourquoi `mouseleave` ?**  
> Si l'utilisateur appuie sur une flèche puis fait glisser la souris hors du bouton sans relâcher le clic, `mouseup` ne se déclenche jamais. `mouseleave` attrape ce cas et envoie quand même `stop` à l'ESP — sinon le robot continuerait indéfiniment.

---
## 4. `updateSensors(d)` — Réception des données MQ-9 `[ESP]` `[UI]`

### Rôle

Appelée automatiquement à chaque fois que l'ESP envoie un paquet de type `"sensor"` via WebSocket, soit environ **toutes les 2 secondes**. Elle met à jour tous les éléments visuels du panneau capteur.

### Structure du JSON reçu de l'ESP

```json
{
  "type":    "sensor",
  "raw":     1842,
  "voltage": "1.482",
  "rs":      "12.340",
  "ratio":   "1.234",
  "ppm":     "47.3",
  "alert":   false
}
```

| Champ | Type | Description |
|-------|------|-------------|
| `type` | `string` | Toujours `"sensor"` pour ce paquet |
| `raw` | `int` 0–4095 | Valeur brute ADC lue sur GPIO34 (résolution 12 bits) |
| `voltage` | `float` | Tension en volts calculée depuis raw (0 – 3.3 V) |
| `rs` | `float` | Résistance interne du capteur RS en kΩ |
| `ratio` | `float` | Rapport RS/RO — utilisé dans la courbe datasheet |
| `ppm` | `float` | Estimation CO en ppm calculée par l'ESP |
| `alert` | `bool` | `true` si le seuil digital (GPIO32) est dépassé |

### Code annoté

```javascript
// Seuils de danger configurables en haut du fichier JS
const MAX_PPM    = 1000;  // Pleine échelle de la barre
const WARN_PPM   = 200;   // Seuil orange
const DANGER_PPM = 500;   // Seuil rouge clignotant

function updateSensors(d) {
  const ppmEl = document.getElementById('ppm');
  const ppm   = parseFloat(d.ppm) || 0;

  // ── Mise à jour des valeurs textuelles ──────────────────────
  document.getElementById('raw').textContent =
    d.raw ?? '—';                                  // Valeur brute ADC

  document.getElementById('voltage').textContent =
    d.voltage ? parseFloat(d.voltage).toFixed(2) : '—';  // 2 décimales

  ppmEl.textContent = ppm.toFixed(1);              // 1 décimale pour les PPM

  // ── Barre de progression PPM ────────────────────────────────
  // La barre utilise un dégradé CSS vert→bleu→orange→rouge
  // La couleur affichée dépend de la largeur (position dans le dégradé)
  const pct = Math.min(ppm / MAX_PPM * 100, 100);
  document.getElementById('ppm-bar').style.width = pct + '%';

  // ── Couleur de la valeur PPM selon le niveau de danger ───────
  if (ppm >= DANGER_PPM)
    ppmEl.className = 'metric-val danger';  // Rouge + animation clignotante
  else if (ppm >= WARN_PPM)
    ppmEl.className = 'metric-val warn';    // Orange
  else
    ppmEl.className = 'metric-val';         // Bleu normal

  // ── Badge d'alerte digital ───────────────────────────────────
  // d.alert = true quand GPIO32 passe LOW
  // Le comparateur interne du module MQ-9 a détecté un dépassement de seuil
  const badge = document.getElementById('alert-badge');
  const txt   = document.getElementById('alert-text');
  if (d.alert) {
    badge.className = 'alert-badge danger';  // Rouge clignotant
    txt.textContent = 'ALERTE GAZ';
  } else {
    badge.className = 'alert-badge';          // Vert normal
    txt.textContent = 'NORMAL';
  }

  // ── Indicateur sur la silhouette SVG ─────────────────────────
  // Le petit cercle sur le corps du robot change de couleur
  const mq9dot = document.getElementById('mq9-dot');
  mq9dot.setAttribute('stroke', d.alert ? '#ff3366' : '#00ff88');
}
```

> **Comment ce JSON arrive-t-il ici ?**  
> Dans `connect()`, le handler `ws.onmessage` reçoit *tous* les messages de l'ESP. Il parse le JSON et regarde le champ `type` : si c'est `"sensor"`, il appelle `updateSensors(d)`. Si c'est `"pong"`, il calcule la latence. C'est un système de routage de messages simple mais efficace.

---
## 5. `activateLegs(cmd)` — Animation de la silhouette `[UI]`

### Rôle

Fonction **purement visuelle** — ne communique pas avec l'ESP. Elle met en évidence les membres actifs sur le SVG du robot pour refléter visuellement le mouvement en cours.

La correspondance commande → membres actifs suit le **trot diagonal** utilisé dans le code C++ : deux membres diagonalement opposés travaillent ensemble, comme un vrai quadrupède.

### Table de correspondance

| Commande | Membres actifs | Logique |
|----------|---------------|----------|
| `forward` | Antérieur droit + Postérieur gauche | Trot diagonal phase 1 |
| `backward` | Postérieur droit + Antérieur gauche | Trot diagonal inversé |
| `left` | Antérieur gauche + Postérieur gauche | Membres côté gauche |
| `right` | Antérieur droit + Postérieur droit | Membres côté droit |
| `stop` | Aucun | Posture neutre |

### Code annoté

```javascript
// Table de correspondance commande → IDs des éléments SVG
const LEG_MAP = {
  forward:  ['leg-fr', 'leg-bl'], // Antérieur droit + Postérieur gauche
  backward: ['leg-br', 'leg-fl'], // Postérieur droit + Antérieur gauche
  left:     ['leg-fl', 'leg-bl'], // Les deux membres gauches
  right:    ['leg-fr', 'leg-br'], // Les deux membres droits
  stop:     []                    // Aucun
};

function activateLegs(cmd) {
  // Remet tous les membres en état "passif" (opacity 35%)
  document.querySelectorAll('.leg').forEach(l => {
    l.classList.remove('active');
    l.classList.add('leg-idle');
  });

  // Active les membres correspondants (opacity 100%)
  const active = LEG_MAP[cmd] || [];
  active.forEach(id => {
    const el = document.getElementById(id);
    if (el) {
      el.classList.add('active');
      el.classList.remove('leg-idle');
    }
  });

  // Met à jour le texte de commande en bas de la vue robot
  document.getElementById('current-cmd').textContent =
    cmd === 'stop' ? 'ARRÊT' : cmd.toUpperCase();
}
```

---
## 6. `sendPing()` — Mesure de la latence réseau `[WS]`

### Rôle

Toutes les 2 secondes, le dashboard envoie un ping à l'ESP. L'ESP répond immédiatement par un pong. La différence de temps donne la **latence aller-retour (RTT)** en millisecondes.

### Code annoté

```javascript
function sendPing() {
  if (ws?.readyState !== 1) return;

  pingTime = Date.now();  // Mémorise l'heure d'envoi
  ws.send(JSON.stringify({ type: 'ping' }));
  // L'ESP répond {"type":"pong"}
  // → ws.onmessage calcule Date.now() - pingTime
}
```

### Côté ESP32 (C++) — réponse au ping

```cpp
if (strcmp(t, "ping") == 0) {
  wsServer.sendTXT(num, "{\"type\":\"pong\"}");
  return;
}
```

### Valeurs de référence

| Latence | Qualité | Impact sur le contrôle |
|---------|---------|------------------------|
| < 20 ms | Excellente | Aucun décalage perceptible |
| 20 – 80 ms | Bonne | Contrôle fluide |
| 80 – 150 ms | Acceptable | Légère inertie |
| > 150 ms | Mauvaise | Décalage visible entre appui et mouvement |

---
## 7. `log(msg, cls)` — Journal des événements `[UI]`

### Rôle

Affiche un message horodaté dans la zone de log en bas du dashboard. Limitée à 40 lignes pour ne pas surcharger le DOM.

### Code annoté

```javascript
function log(msg, cls = 'entry') {
  const el  = document.getElementById('log');
  const now = new Date().toLocaleTimeString('fr-FR'); // ex: "14:32:05"

  // Classes CSS disponibles :
  //   'entry' → texte blanc  (événements généraux)
  //   'cmd'   → texte cyan   (commandes envoyées à l'ESP)
  //   'warn'  → texte orange (avertissements)
  //   'err'   → texte rouge  (erreurs de connexion)
  el.innerHTML += `<div class="entry ${cls}">[${now}] ${msg}</div>`;

  el.scrollTop = el.scrollHeight; // Auto-scroll vers le bas

  // Limite à 40 lignes pour éviter de surcharger le DOM
  while (el.children.length > 40) el.removeChild(el.firstChild);
}
```

---
## 8. Contrôle au clavier `[UI]`

### Rôle

En plus des boutons à l'écran, le dashboard écoute les touches du clavier. La logique est identique : `keydown` démarre la commande, `keyup` l'arrête — exactement comme `mousedown` / `mouseup`.

### Mapping clavier

| Touche | Commande |
|--------|----------|
| `↑` (ArrowUp) | `forward` |
| `↓` (ArrowDown) | `backward` |
| `←` (ArrowLeft) | `left` |
| `→` (ArrowRight) | `right` |
| Espace | `stop` |

### Code annoté

```javascript
const KEY_CMD = {
  'ArrowUp':    'forward',
  'ArrowDown':  'backward',
  'ArrowLeft':  'left',
  'ArrowRight': 'right',
  ' ':          'stop'     // Barre espace
};

document.addEventListener('keydown', e => {
  const cmd = KEY_CMD[e.key];
  if (!cmd) return;
  e.preventDefault(); // Empêche le scroll de la page avec les flèches
  const btn = document.querySelector(`.btn[data-cmd="${cmd}"]`);
  if (btn) startCmd(cmd, btn); // Réutilise exactement la même logique
});

document.addEventListener('keyup', e => {
  const cmd = KEY_CMD[e.key];
  if (cmd && cmd !== 'stop') stopCmd();
});
```

---
## 9. `setStatus(ok)` — Indicateur de connexion `[UI]`

### Rôle

Met à jour le point de statut et le texte dans l'en-tête selon l'état de la connexion WebSocket.

### Code annoté

```javascript
function setStatus(ok) {
  // Le point coloré dans le header
  document.getElementById('dot').className =
    'dot' + (ok ? ' connected' : '');
  //   connected → vert avec animation de pulsation CSS
  //   (rien)    → rouge fixe

  // Le texte à côté du point
  document.getElementById('ws-status').textContent =
    ok ? 'CONNECTÉ' : 'DÉCONNECTÉ';
  document.getElementById('ws-status').className =
    ok ? 'connected' : '';

  // Remet la latence à "—" si déconnecté
  if (!ok) document.getElementById('latency').textContent = '— ms';
}
```

---
## 10. Flux complet d'un mouvement — de bout en bout

Ce schéma résume ce qui se passe à chaque niveau du système quand l'utilisateur appuie sur la flèche **avant** :

```
Utilisateur appuie sur ↑
        │
        ▼
  [Événement mousedown / keydown]
        │
        ▼
  startCmd('forward', btn)
  ├── btn.classList.add('pressed')     → bouton devient bleu
  ├── activateLegs('forward')          → SVG allume FR + BL
  ├── sendCmd('forward')               → 1er envoi JSON
  └── setInterval(sendCmd, 150ms)      → répétition continue
        │
        │  ws.send({type:'cmd', cmd:'forward', speed:50})
        │  ────────────────────────── WiFi ──────────────────────────►
        │
        ▼
  ESP32 — onWsEvent()
  └── currentCmd = CMD_FORWARD
        │
        ▼
  ESP32 — loop() → tickGait()
  └── applyFrame(GAIT_FORWARD[frame % 4])
        │
        ▼
  PCA9685 — I2C
  └── ch0, ch1, ch2, ch3, ch4, ch5, ch6, ch7 → positions PWM
        │
        ▼
  8 servo-moteurs → MOUVEMENT


Utilisateur relâche ↑
        │
        ▼
  [Événement mouseup / keyup]
        │
        ▼
  stopCmd()
  ├── clearInterval(sendInterval)      → arrêt de la répétition
  ├── classList.remove('pressed')      → bouton revient normal
  ├── activateLegs('stop')             → SVG éteint tous les membres
  └── sendCmd('stop')                  → envoi final
        │
        │  ws.send({type:'cmd', cmd:'stop', speed:50})
        │  ────────────────────────── WiFi ──────────────────────────►
        │
        ▼
  ESP32 — currentCmd = CMD_STOP
  └── standStill() → tous les servos en position neutre
```

---

## Résumé des fonctions

| Fonction | Tag | Rôle résumé |
|----------|-----|-------------|
| `connect()` | `[WS]` `[UI]` | Ouvre la connexion WebSocket et attache les handlers |
| `sendCmd(cmd)` | `[WS]` | Construit et envoie un JSON de commande moteur |
| `startCmd(cmd, el)` | `[UI]` `[WS]` | Déclenche le mouvement et lance la répétition |
| `stopCmd()` | `[UI]` `[WS]` | Arrête le mouvement et envoie stop à l'ESP |
| `updateSensors(d)` | `[ESP]` `[UI]` | Met à jour le panneau MQ-9 depuis les données reçues |
| `activateLegs(cmd)` | `[UI]` | Anime la silhouette SVG selon la direction |
| `sendPing()` | `[WS]` | Mesure la latence aller-retour WiFi |
| `setStatus(ok)` | `[UI]` | Met à jour l'indicateur de connexion dans le header |
| `log(msg, cls)` | `[UI]` | Ajoute une entrée horodatée dans le journal |